#### General guidance

This serves as a template which will guide you through the implementation of this task. It is advised
to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [42]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

# Add any additional imports here (however, the task is solvable without using 
# any additional imports)
# import ...

 #### Loading data

In [43]:
data = pd.read_csv("train.csv")
y = data["y"].to_numpy()
data = data.drop(columns="y")
# print a few data samples
print(data.head())

         x1   x2     x3   x4     x5     x6     x7      x8    x9    x10   x11  \
0   0.06724  0.0   3.24  0.0  0.460  6.333   17.2  5.2146   4.0  430.0  16.9   
1   9.23230  0.0  18.10  0.0  0.631  6.216  100.0  1.1691  24.0  666.0  20.2   
2   0.11425  0.0  13.89  1.0  0.550  6.373   92.4  3.3633   5.0  276.0  16.4   
3  24.80170  0.0  18.10  0.0  0.693  5.349   96.0  1.7028  24.0  666.0  20.2   
4   0.05646  0.0  12.83  0.0  0.437  6.232   53.7  5.0141   5.0  398.0  18.7   

      x12    x13  
0  375.21   7.34  
1  366.15   9.53  
2  393.74  10.50  
3  396.90  19.77  
4  386.40  12.34  


#### Calculating the average RMSE

In [44]:
def calculate_RMSE(w, X, y):
    """This function takes test data points (X and y), and computes the empirical RMSE of 
    predicting y from X using a linear model with weights w. 

    Parameters
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression 
    X: matrix of floats, dim = (15,13), inputs with 13 features
    y: array of floats, dim = (15,), input labels

    Returns
    ----------
    rmse: float: dim = 1, RMSE value
    """
    rmse = 0
    n = len(w)

    yhat = X@w
    error = np.subtract(y, yhat)
    sum = np.dot(error, error)

    rmse =  (sum / n) ** 0.5
    
    assert np.isscalar(rmse)
    return rmse

#### Fitting the regressor

In [60]:
def fit(X, y, lam):
    """
    This function receives training data points, then fits the ridge regression on this data
    with regularization hyperparameter lambda. The weights w of the fitted ridge regression
    are returned. 

    Parameters
    ----------
    X: matrix of floats, dim = (135,13), inputs with 13 features
    y: array of floats, dim = (135,), input labels
    lam: float. lambda parameter, used in regularization term

    Returns
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression
    """
    weights = np.zeros((13,))
    A = X.transpose()@X + lam * np.eye(13)
    b = X.transpose() @ y
    weights = np.linalg.solve(A, b)
    assert weights.shape == (13,)
    return weights

#### Performing computation

In [65]:
"""
Main cross-validation loop, implementing 10-fold CV. In every iteration 
(for every train-test split), the RMSE for every lambda is calculated, 
and then averaged over iterations.

Parameters
---------- 
X: matrix of floats, dim = (150, 13), inputs with 13 features
y: array of floats, dim = (150, ), input labels
lambdas: list of floats, len = 5, values of lambda for which ridge regression is fitted and RMSE estimated
n_folds: int, number of folds (pieces in which we split the dataset), parameter K in KFold CV

Compute
----------
avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
"""
X = data.to_numpy()
# The function calculating the average RMSE
lambdas = [0.1, 1, 10, 100, 200]
n_folds = 10
chunk_size = np.ceil((X.size / n_folds + 0.5))

RMSE_mat = np.zeros((n_folds, len(lambdas)))
for j in range(len(lambdas)):
    lam = lambdas[j]
    for i in range( n_folds):
        
        foldstart = X[: i * 10 ]
        foldend = X[(i+1)*10 :]
        val_X = X[i * 10:(i+1)*10 ]
        train_X = np.concatenate((foldstart , foldend))

        ystart = y[:i*10]
        yend = y[(i+1)*10:]
        val_y = y[i * 10:(i+1)*10 ]
        train_y = np.concatenate((ystart,yend))

        w = fit(train_X,train_y,lam)
        rmse = calculate_RMSE(w,val_X,val_y)
        RMSE_mat[i,j] = rmse

    

# TODO: Enter your code here. Hint: Use functions 'fit' and 'calculate_RMSE' with training and test data
# and fill all entries in the matrix 'RMSE_mat'

avg_RMSE = np.mean(RMSE_mat, axis=0) # avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
assert avg_RMSE.shape == (5,)

In [66]:
RMSE_mat

array([[7.33708152, 7.31368597, 7.17063498, 7.13004019, 7.19963207],
       [3.53425721, 3.6478874 , 4.02551934, 5.31219205, 5.83606497],
       [5.35081765, 5.0879414 , 4.64172841, 3.50209599, 3.39120969],
       [7.93176855, 7.90852038, 7.9068262 , 7.81632064, 7.84662665],
       [3.76548405, 3.75114384, 3.53481978, 3.63632367, 3.78485145],
       [3.68362844, 3.67528215, 3.74778037, 4.62586045, 5.03170197],
       [3.37378626, 3.42828643, 3.56158401, 4.34749777, 4.77847103],
       [3.81384709, 3.76742907, 3.9028475 , 4.87251815, 5.3087735 ],
       [4.26115737, 4.31600365, 4.52819672, 5.89275419, 6.52633244],
       [6.19824117, 6.24795288, 6.33638811, 6.99216267, 7.3387808 ]])

In [67]:
# Save results in the required format
np.savetxt("./results.csv", avg_RMSE, fmt="%.12f")